In [1]:
import pandas as pd
import numpy as np


In [2]:
df=pd.read_csv('dataset/synthetic_logs.csv')
df

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert
...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert


In [3]:
df.source.unique()

<StringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [4]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize

df['log_message'] = df['log_message'].astype(str)

model = SentenceTransformer('all-MiniLM-L6-v2')

log_messages = df['log_message'].tolist()
embeddings = model.encode(log_messages, show_progress_bar=True)

embeddings = normalize(embeddings)

dbscan = DBSCAN(eps=0.3, min_samples=1, metric='cosine')
clusters = dbscan.fit_predict(embeddings)

df['cluster'] = clusters



f:\cuisine recommender web app\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1855.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 76/76 [01:24<00:00,  1.11s/it]


In [5]:
df[df.cluster==1]
cluster_counts=df['cluster'].value_counts()
large_clusters=cluster_counts[cluster_counts>10].index
for cluster in large_clusters:
    print(f'Cluster{cluster}')
    print(df[df['cluster']==cluster]['log_message'].head(5).to_string(index=False))

Cluster0
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...
nova.osapi_compute.wsgi.server [req-f0bffbc3-5a...
nova.osapi_compute.wsgi.server [req-2bf7cfee-a2...
Cluster7
Multiple bad login attempts detected on user 85...
Alert: brute force login attempt from 192.168.8...
Suspicious login activity detected from 192.168...
Denied access attempt on restricted account Acc...
Abnormal system behavior on server 40, potentia...
Cluster9
Account with ID 5351 created by User634.
                User User685 logged out.
System reboot initiated by user User243.
                 User User395 logged in.
                 User User225 logged in.
Cluster5
nova.compute.claims [req-a07ac654-8e81-416d-bfb...
nova.compute.resource_tracker [req-addc1839-2ed...
nova.compute.claims [req-d6986b54-3735-4a42-907...
nova.compute.claims [req-72b4858f-049e-49e1-b31...
nova.compute.claims [req-5c8f52bd-8e3c-41f0-9

In [6]:
import re

def classify_with_regex(log_message):
    # Regex patterns with their labels
    regex_patterns = {
        r"User User\d+ logged (in|out)\.": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully\.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully\.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }

    # Iterate over patterns
    for pattern, label in regex_patterns.items():
        # Use re.search to match anywhere in the string and ignore case
        if re.search(pattern, log_message, re.IGNORECASE):
            return label

    # Return a string, not a tuple
    return None

# Example usage
logs = [
    "User User123 logged in.",
    "Backup started at 16:00",
    "Some unrelated log message"
]

for log in logs:
    print(f"{log} => {classify_with_regex(log)}")

User User123 logged in. => User Action
Backup started at 16:00 => System Notification
Some unrelated log message => None


In [7]:
classify_with_regex('User User123 logged in.')

'User Action'

In [8]:
df['regex_label']=df['log_message'].apply(classify_with_regex)

In [9]:
df['regex_label'].nunique()

2

In [10]:
df_non_regex=df[df.regex_label.isna()]

In [11]:
#we have quite a few rows left where we cannot use regex because they are unknown there so well use either bert or llm to classify them into a category

#for the categories which have less training examples we will use llm , for the rest we will use bert
print(df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts()<=5].index.tolist())

['Workflow Error', 'Deprecation Warning']


In [12]:
#the categories with 4 and 3 examples which are workflow error and depreciation warning , we will use llm to classify them
df_non_legacy=df_non_regex[df_non_regex.source!='LegacyCRM']
df_non_legacy.source.unique()
# we will be using bert for this dataset

<StringArray>
['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI']
Length: 5, dtype: str

In [13]:
df_non_legacy['log_message'] = df_non_legacy['log_message'].astype(str)
filtered_embeddings=model.encode(df_non_legacy['log_message'].tolist())

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
y=df_non_legacy['target_label']
X_train,X_test,y_train,y_test=train_test_split(filtered_embeddings,y,test_size=0.2)
clf=LogisticRegression(max_iter=100)
clf.fit(X_train,y_train)
y_pred=clf.predict(X_test)
report=classification_report(y_test,y_pred)
print(report)

                precision    recall  f1-score   support

Critical Error       0.97      1.00      0.98        31
         Error       0.97      0.97      0.97        32
   HTTP Status       1.00      1.00      1.00       199
Resource Usage       1.00      1.00      1.00        36
Security Alert       1.00      0.99      0.99        83

      accuracy                           0.99       381
     macro avg       0.99      0.99      0.99       381
  weighted avg       0.99      0.99      0.99       381



In [16]:
import joblib
joblib.dump(clf,'models/log_classifier.joblib')

['models/log_classifier.joblib']